# 🌱 Phase 3 — LLM : 토마토 재배 도우미

**예방 코치 + 병충해 확인 + 자연어 처방** — Phase 1(ML)·2(DL)의 예측/진단을 *사람 말*로 완성한다.

| 항목 | 내용 |
|---|---|
| **LLM** | Ollama 로컬 구동 · `qwen2.5:14b` (한국어·function calling 안정) |
| **RAG** | 농사로(nongsaro.go.kr) 재배·방제 가이드 4종 · 임베딩 `bge-m3` |
| **분업 원칙(불변)** | 진단은 ML/DL, **설명·처방·코칭은 LLM** (LLM에 진단 시키지 않음 → 환각 차단) |
| **알림** | 디스코드 Webhook (조기경보·긴급) |

> 이 노트북은 `src/llm/` 실제 구현을 그대로 import해 시연합니다. (앱 서비스 코드가 진실 소스)
> Ollama 데몬(`ollama serve`)과 `qwen2.5:14b`·`bge-m3` pull이 되어 있으면 셀이 실제로 실행됩니다.


## 0. 아키텍처 — 재배 도우미 loop

DL 진단을 *반응형 진단기*가 아니라 **"확인 단추"** 로 재정의 → 예방·확인·처방·상담이 한 흐름.

```
[평상시]  A. 일일 코치 ──── 매일 "오늘 할 일" (LSTM 환경 예측)
              │
[이상 징후] B. 조기 경보 ──── 환경 × 질병위험 → "지금 잎 확인해봐요"
              ▼
[확인]    🔬 DL 진단 ──────── 잎 사진 → 병충해 맞는지 확인 (CNN·게이트·YOLO)
              ├─ 정상 → C. 안심 "괜찮아요, 물자국이에요"
              └─ 질병 → C. 처방 + 용어풀이 + 📖 농사로 RAG 근거
[후속]    D. 대화형 Q&A ───── "약 없이 환경으로만 잡을 수 있어요?"
```

**DL = 확인 단추 / LLM = 그 앞(예방·경보)과 뒤(처방·안심·상담)를 채우는 도우미.**


## 1. 환경 셋업

`src/`를 경로에 추가하고 `.env`(모델명·RAG 백엔드)를 로드한다.
> ⚠️ macOS 로컬은 torch+xgboost libomp 세그폴트가 있어 커널을 `OMP_NUM_THREADS=1`로 띄우거나
> 노트북 최상단에서 환경변수를 설정해야 한다(서버는 무관).

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")  # macOS libomp 세그폴트 회피(서버는 불필요)

import sys, json
from pathlib import Path

# 프로젝트 루트 탐색 (이 노트북이 notebooks/ 또는 제출 zip 어디에 있든 src를 찾도록)
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src" / "llm").exists():
        break
    ROOT = ROOT.parent
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))
print("ROOT :", ROOT)
print("src/llm 존재:", (SRC / "llm").exists())

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)
print("OLLAMA_MODEL :", os.getenv("OLLAMA_MODEL", "qwen2.5:14b (기본)"))
print("RAG_BACKEND  :", os.getenv("RAG_BACKEND", "memory (기본)"))


ROOT : /Users/jeongjaebong/IntelliJ/mycode/toy_project/solo/smartfarm_ai
src/llm 존재: True
OLLAMA_MODEL : qwen2.5:14b
RAG_BACKEND  : memory (기본)


In [2]:
# Ollama 데몬 가동 여부 확인 — 이후 LLM 셀 실행 가능한지 판단
import urllib.request
def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
            names = [m["name"] for m in json.load(r)["models"]]
            return True, names
    except Exception as e:
        return False, str(e)

OLLAMA_OK, MODELS = ollama_up()
print("Ollama 가동:", OLLAMA_OK)
print("설치 모델 :", MODELS if OLLAMA_OK else "(데몬 꺼짐 — LLM 셀은 스킵되고 코드만 표시됨)")


Ollama 가동: True
설치 모델 : ['bge-m3:latest', 'qwen2.5:14b']


## 2. Function Calling — DL 출력을 구조화 함수로 안정 전달

ML/DL 결과를 자연어로 풀어 넣지 않고 **tool 호출 → 실제 추론 실행 → 구조화 결과 반환**.
LLM은 진단을 *하지 않고* tool을 *호출만* 한다(분업 원칙).

| tool | 실제 실행 | 반환 |
|---|---|---|
| `get_diagnosis` | OOD 게이트→부위 게이트→CNN 진단 | label·prob·probs·ood_blocked |
| `get_detection` | YOLO 병변 검출 | boxes·lesion_count |
| `get_forecast` | LSTM 환경 예측 | next_temp·trend·humidity_risk |
| `get_weather` | 기상청(KMA) API | 실황/3일 예보 |


In [3]:
from llm.tools import TOOL_SCHEMAS, TOOL_REGISTRY, get_diagnosis, get_forecast

print("LLM에 노출되는 tool:", list(TOOL_REGISTRY))
# get_diagnosis 스키마(모델이 보는 설명)
print(json.dumps(TOOL_SCHEMAS[0], ensure_ascii=False, indent=2))


LLM에 노출되는 tool: ['get_diagnosis', 'get_detection', 'get_forecast', 'get_weather']
{
  "type": "function",
  "function": {
    "name": "get_diagnosis",
    "description": "토마토 잎 사진을 진단한다(잎마름역병 late_blight/잎곰팡이병 leaf_mold/정상 normal/tylcv 4종). 잎이 아니면 진단 대신 차단 사유를 돌려준다.",
    "parameters": {
      "type": "object",
      "properties": {
        "image_path": {
          "type": "string",
          "description": "진단할 잎 사진 파일 경로"
        }
      },
      "required": [
        "image_path"
      ]
    }
  }
}


In [4]:
# tool 직접 실행 데모 ① — 잎 사진 진단(게이트 내장)
sample = sorted((ROOT / "data/tomato/val/leaf_mold").glob("*.jpg"))
if sample:
    diag = get_diagnosis(str(sample[0]))
    print("입력:", sample[0].name)
    print(json.dumps(diag, ensure_ascii=False, indent=2))
else:
    print("샘플 이미지 없음 — data/tomato/val/leaf_mold 확인")


입력: leaf_mold_V006_77_1_18_11_03_13_1_3248b_20201102_26.jpg.jpg
{
  "ood_blocked": false,
  "label": "leaf_mold",
  "label_kr": "잎곰팡이병",
  "prob": 0.992,
  "probs": {
    "late_blight": 0.001,
    "leaf_mold": 0.992,
    "normal": 0.007,
    "tylcv": 0.0
  },
  "part": "leaf"
}


In [5]:
# tool 직접 실행 데모 ② — LSTM 환경 예측(고습이면 곰팡이 위험↑ → 선제 처방 근거)
fc = get_forecast()
print(json.dumps(fc, ensure_ascii=False, indent=2))


{
  "next_temp": 16.4,
  "recent_temp": 16.2,
  "trend": "유지",
  "humidity_risk": "높음",
  "humidity_mean": 99.5
}


## 3. RAG — 농사로 재배가이드로 처방 근거 확보

진단 라벨로 스코프한 검색(임베딩 `bge-m3` · 코사인 유사도) → 처방에 **출처 인용**.
근거에 없는 약제명·수치는 지어내지 않게 하는 사실 기반 장치.

코퍼스: 잎곰팡이병·잎마름역병·tylcv·토마토 일반재배 4문서(`data/nongsaro/*.md`).

In [6]:
from llm.rag import retrieve

if OLLAMA_OK:
    chunks = retrieve("잎에 곰팡이가 피고 노랗게 변해요", disease="leaf_mold", k=2)
    for c in chunks:
        print(f"[{c['title']}] (score={c.get('score'):.3f})  출처={c.get('source_name')}")
        print("  ", c["text"][:120], "...\n")
else:
    print("Ollama 꺼짐 — retrieve()는 bge-m3 임베딩이 필요해 스킵")


[토마토 잎곰팡이병(Leaf mold) 발생환경·증상·방제] (score=0.730)  출처=국가농작물병해충관리시스템(NCPMS·농촌진흥청)
   잎에 발생한다.
처음에는 잎의 표면에 흰색 또는 담회색의 반점으로 나타나고 진전되면 황갈색 병반으로 확대된다.
잎 뒷면에 담갈색의 병반이 형성되는데, 병반상에는 갈색의 곰팡이가 융단처럼 밀생되어 있는 것을 볼 수 있 ...

[토마토 잎곰팡이병(Leaf mold) 발생환경·증상·방제] (score=0.510)  출처=국가농작물병해충관리시스템(NCPMS·농촌진흥청)
   병원균은 병든 잎이나 종자 등에서 겨울을 지내고 1차 전염원이 되나, 시설재배에서는 병원균이 각종 농자재에 붙어 겨울을 지내기도 한다.
2차 전염은 병반상에 형성된 포자가 전반되어 잎의 기공을 통하여 침입하여 발병된 ...



## 4. 환각 방어 3종 ★ — 신뢰성 핵심

DL이 주는 *확률·게이트·클래스 목록*을 활용해 "모를 땐 모른다고" 말하게 한다.

1. **신뢰도 톤 분기** — `prob` 구간별 강도(>0.8 단정 / 0.6~0.8 가능성 / <0.6 정밀확인)
2. **게이트 차단 안내** — 차단 시 *왜·어떻게 다시 찍을지* 안내
3. **클래스 한정성** — 아는 범위(잎병 4종)를 system 프롬프트에 고정, 밖이면 "진단 보류"


In [7]:
from llm.prescribe import _guard_directive

# 확률 구간별로 톤 지시문이 어떻게 갈리는지
for case in [
    {"label": "leaf_mold", "prob": 0.92, "ood_blocked": False},   # 고확신
    {"label": "leaf_mold", "prob": 0.68, "ood_blocked": False},   # 애매
    {"label": "leaf_mold", "prob": 0.41, "ood_blocked": False},   # 저확신
    {"ood_blocked": True, "reason": "잎이 아닌 부위로 판정(과실)"}, # 게이트 차단
]:
    print("입력:", {k: case[k] for k in case})
    print("→ 지시문:", _guard_directive(case), "\n")


입력: {'label': 'leaf_mold', 'prob': 0.92, 'ood_blocked': False}
→ 지시문: 진단 신뢰도 92% — 확신이 높다. 병명을 단정하고 즉시 방제를 안내하라. 아는 범위는 잎 병해 4종뿐임을 잊지 마라. 

입력: {'label': 'leaf_mold', 'prob': 0.68, 'ood_blocked': False}
→ 지시문: 진단 신뢰도 68% — 확신이 중간이다. 단정하지 말고 '가능성'으로 표현하며 관찰을 권하라. 아는 범위는 잎 병해 4종뿐임을 잊지 마라. 

입력: {'label': 'leaf_mold', 'prob': 0.41, 'ood_blocked': False}
→ 지시문: 진단 신뢰도 41% — 확신이 낮다. 절대 단정하지 말고 정밀 확인(재촬영·전문가 상담)을 우선 권하라. 아는 범위는 잎 병해 4종뿐임을 잊지 마라. 

입력: {'ood_blocked': True, 'reason': '잎이 아닌 부위로 판정(과실)'}
→ 지시문: 진단 차단(사유: 잎이 아닌 부위로 판정(과실)). 병명을 절대 단정하지 말라. 이유를 쉽게 설명하고 '재촬영시점'에 잎 뒷면을 밝은 곳에서 다시 찍는 방법을 담아라. 



## 5. 구조화 출력 — Prescription 스키마 강제

최종 처방은 자유 텍스트가 아니라 **고정 JSON 스키마**로 강제(`format=`).
Streamlit 앱·디스코드 알림이 같은 소스를 파싱해 쓴다. 스키마 위반 시 1회 재시도 후 안전 폴백.

In [8]:
from llm.prescribe import Prescription

# 모델에 전달되는 스키마(각 필드 설명이 작성 가이드로 실림)
print(json.dumps(Prescription.model_json_schema()["properties"], ensure_ascii=False, indent=2))


{
  "진단요약": {
    "description": "진단 결과를 한 문장으로. 예: '잎곰팡이병으로 보입니다(신뢰도 87%)'",
    "title": "진단요약",
    "type": "string"
  },
  "원인": {
    "description": "이 병이 왜 생기는지 초보자도 알기 쉽게 설명한 완전한 문장",
    "title": "원인",
    "type": "string"
  },
  "즉시조치": {
    "description": "지금 당장 해야 할 구체적 조치",
    "title": "즉시조치",
    "type": "string"
  },
  "예방": {
    "description": "앞으로 재발을 막기 위한 관리 방법",
    "title": "예방",
    "type": "string"
  },
  "재촬영시점": {
    "description": "언제 다시 사진을 찍어 확인하면 좋은지, 또는 재촬영 방법",
    "title": "재촬영시점",
    "type": "string"
  },
  "근거출처": {
    "description": "처방 근거 출처(RAG 검색 결과로 코드가 채우므로 모델은 비워도 됨)",
    "items": {
      "type": "string"
    },
    "title": "근거출처",
    "type": "array"
  }
}


## 6. 일일 코치(A) · 조기 경보(B) — 사진 없는 흐름

환경 예측(LSTM)만으로 평상시 코칭과 병해 조기경보를 생성. 경보는 이력에 저장(서버=pgvector).

In [9]:
from llm import pipeline

if OLLAMA_OK:
    print("=== 🌅 일일 코치 (A) ===")
    coach = pipeline.daily_coach()
    print(json.dumps(coach.model_dump(), ensure_ascii=False, indent=2))
else:
    print("Ollama 꺼짐 — LLM 생성 스킵 (daily_coach는 qwen2.5:14b 필요)")


=== 🌅 일일 코치 (A) ===


{
  "요약": "오늘은 토마토의 건강을 위해 과습 문제를 해결하는 것이 긴요합니다.",
  "오늘_할일": [
    "주변 환경을 점검하고 습도가 너무 높지 않도록 조절하세요. 통풍이 잘 되게 창문을 열거나 제습기를 사용해주세요."
  ],
  "근거": "토마토는 습한 환경에서 곰팡이나 병에 쉽게 걸리기 때문입니다. 습도가 90%를 넘으면 건강 문제로 이어질 수 있습니다. 또한, 온도와 습도 조합이 적절하게 유지되어야 토마토의 성장과 생산성이 최적화됩니다."
}


In [10]:
if OLLAMA_OK:
    print("=== ⚠️ 조기 경보 (B) — 습도위험 높으면 RAG 잎곰팡이병 근거 결합 ===")
    warn = pipeline.early_warning()
    print(json.dumps(warn.model_dump(), ensure_ascii=False, indent=2))
else:
    print("Ollama 꺼짐 — 스킵")


=== ⚠️ 조기 경보 (B) — 습도위험 높으면 RAG 잎곰팡이병 근거 결합 ===


{
  "경보수준": "높음",
  "위험병해": "흰 반점 병해 (가지송진혹bine canker, 흰 반점rust 등이 의심됨)",
  "이유": "최근 습도 평균이 99.5%로 매우 높아서 병원균의 번식과 확산에 이상적인 조건을 제공하고 있다.",
  "권장조치": "- 병든 잎 제거\n- 상대습도를 90% 이하로 관리\n- 통풍개선 및 밀식 방지\n- 질소질 비료의 과용 피하기"
}


## 7. 자연어 처방(C) — 전체 오케스트레이션

`prescribe()` = LLM 주도 function calling(agentic). 잎 사진 → LLM이 `get_diagnosis` tool 호출
→ 실제 CNN 진단 → 환각방어 지시문 주입 → RAG 근거 결합 → LSTM 교차 → 구조화 처방 생성.

> 서버 처방 버튼은 `prescribe_fast()`(1-call, writer=exaone3.5:2.4b)로 342.6s→16.2s 최적화.
> 여기선 전체 흐름이 보이는 agentic `prescribe()`를 시연.

In [11]:
from llm.prescribe import prescribe

if OLLAMA_OK and sample:
    print(f"[입력 이미지] {sample[0].name}\n처방 생성 중(10~15s)...\n")
    presc = prescribe("이 토마토 잎 사진 좀 봐줘. 병이면 어떻게 조치해야 해?",
                      image_path=str(sample[0]))
    print(json.dumps(presc.model_dump(), ensure_ascii=False, indent=2))
else:
    print("Ollama 꺼짐 또는 샘플 없음 — 처방 생성 스킵")


[입력 이미지] leaf_mold_V006_77_1_18_11_03_13_1_3248b_20201102_26.jpg.jpg
처방 생성 중(10~15s)...



{
  "진단요약": "잎곰팡이병",
  "원인": "90% 이상의 높은 습도와 어두운 환경에서 발생합니다.",
  "즉시조치": "습도를 낮추어 주세요. 창문을 열거나 제습기를 활용해 보세요.",
  "예방": "통풍이 잘 되게 하고, 밀식하지 않도록 주의하세요. 질소질 비료 과용 피합니다.",
  "재촬영시점": "잎 사진을 계속 체크하고 이상 징후 발견 시 재촬영해주세요.",
  "근거출처": [
    "토마토 잎곰팡이병(Leaf mold) 발생환경·증상·방제 (국가농작물병해충관리시스템(NCPMS·농촌진흥청)) — https://www.data.go.kr/data/15002034/openapi.do"
  ]
}


## 8. 정리

| 기법 | 구현 위치 | 역할 |
|---|---|---|
| Function calling | `src/llm/tools.py` | DL 추론을 tool로 안정 연결(분업) |
| RAG | `src/llm/rag/` | 농사로 가이드 근거 검색(bge-m3) |
| 환각 방어 3종 | `src/llm/prescribe.py` `_guard_directive` | 신뢰도·게이트·범위 한정 |
| 구조화 출력 | `Prescription` 스키마 | JSON 강제 → 앱/알림 공유 |
| 시간축 처방 | `_forecast_directive` | 진단 × LSTM 교차 선제 조치 |

**배운 점** — LLM에 진단을 맡기지 않고 *설명·처방*에만 쓰는 분업 + 환각방어 3종이
초보자용 재배 도우미의 신뢰성을 좌우했다. 로컬(`qwen2.5:14b`)로 비용 0·오프라인 구동.
